In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: Isang Qiskit Function ng ColibriTD
*Tingnan ang [API reference](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Ang Qiskit Functions ay isang experimental na feature na available para sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.
## Pangkalahatang-ideya
Ang Partial Differential Equation (PDE) solver na ipinapakita dito ay bahagi ng aming Quantum Innovative Computing Kit (QUICK) platform (QUICK-PDE), at nakabalot bilang isang Qiskit Function. Sa pamamagitan ng QUICK-PDE function, maaari kang magsulsi ng domain-specific partial differential equations sa IBM Quantum QPUs. Ang function na ito ay nakabatay sa algorithm na inilalarawan sa [papel ng paglalarawan ng H-DES ng ColibriTD](https://arxiv.org/abs/2410.01130). Kaya ng algorithm na ito na harapin ang mga kumplikadong multi-physics na problema, simula sa Computational Fluid Dynamics (CFD) at Materials Deformation (MD), at may mga karagdagang use case na paparating pa.

Para harapin ang mga differential equations, ang mga trial solution ay naka-encode bilang linear combinations ng mga orthogonal function (karaniwang Chebyshev polynomials, at mas partikular na $2^n$ sa kanila kung saan ang $n$ ay ang bilang ng mga qubit na nag-e-encode ng iyong function), na may parameter na mga anggulo ng isang Variable Quantum Circuit (VQC). Ang ansatz ay bumubuo ng isang estado na nag-e-encode ng function, na sinusuri ng mga observable na ang kombinasyon nito ay nagbibigay-daan sa pag-evaluate ng function sa lahat ng punto. Maaari kang mag-evaluate ng loss function kung saan naka-encode ang mga differential equations, at i-fine-tune ang mga anggulo sa isang hybrid loop, tulad ng ipinapakita sa sumusunod. Ang mga trial solution ay unti-unting humahantong sa mga aktwal na solusyon hanggang sa makamit mo ang kasiya-siyang resulta.

![Daloy ng trabaho ng QUICK-PDE function](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Bukod sa hybrid loop na ito, maaari ka ring mag-chain ng iba't ibang optimizer. Ito ay kapaki-pakinabang kapag gusto mong gamitin ang isang global optimizer para mahanap ang magandang set ng mga anggulo, at pagkatapos ay isang mas pinong optimizer para sundan ang gradient patungo sa pinakamahusay na set ng kalapit na mga anggulo. Sa kaso ng computational fluid dynamics (CFD), ang default na optimization sequence ay nagbibigay ng pinakamahusay na resulta — ngunit sa kaso ng material deformation (MD), habang nagbibigay ng magandang resulta ang default, maaari mo itong i-configure pa para sa mga benepisyong partikular sa problema.

Tandaan na para sa bawat variable ng function, tinutukoy natin ang bilang ng mga qubit (na maaari mong baguhin). Sa pag-stack ng 10 magkaparehong circuit at pag-evaluate ng 10 magkaparehong observable sa iba't ibang qubit sa buong isang malaking circuit, maaari kang mag-noise-mitigate sa loob ng proseso ng CMA optimization, umaasa sa noise learner method, at makabuluhang mabawasan ang bilang ng mga shot na kailangan.
### Computational fluid dynamics
Ang inviscid Burgers' equation ay nagmo-modelo ng daloy ng mga di-malapot na likido tulad ng sumusunod:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

Ang $u$ ay kumakatawan sa field ng bilis ng likido. Ang use case na ito ay may temporal na boundary condition: maaari kang pumili ng initial condition at hayaan ang sistema na mag-relax. Sa kasalukuyan, ang mga tanging tinatanggap na initial condition ay mga linear function: $ax + b$.

Ang mga argumento para sa mga differential equation ng CFD ay nasa isang fixed grid, tulad ng sumusunod:

- Ang $t$ ay nasa pagitan ng 0 at 0.95 na may 30 sample point. Ang $x$ ay nasa pagitan ng 0 at 0.95 na may step size na 0.2375.

### Material Deformation
Ang use case na ito ay nakatuon sa hypoelastic deformation na may one-dimensional tensile test, kung saan ang isang bar na nakaayos sa espasyo ay hinihila sa kabilang dulo nito. Inilalarawan namin ang problema tulad ng sumusunod:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

Ang $K$ ay kumakatawan sa bulk modulus ng materyal na inuuntog, ang $n$ ang exponent ng isang power law, ang $b$ ang puwersa bawat unit mass, ang $\epsilon_0$ ang proportional stress limit, ang $\sigma_0$ ang proportional strain limit, ang $u$ ang stress function, at ang $\sigma$ ang strain function.

Ang isinasaalang-alang na bar ay may unitary na haba. Ang use case na ito ay may boundary condition para sa surface stress $t$, o ang dami ng trabahong kailangan para maunat ang bar.

Ang mga argumento para sa mga differential equation ng MD ay nasa isang fixed grid, tulad ng sumusunod:

- Ang $x$ ay nasa pagitan ng 0 at 1 na may step size na 0.04.
## Mga Benchmark
Ang sumusunod na talahanayan ay nagpapakita ng mga istatistika sa iba't ibang run ng aming function.

| Halimbawa                    | Bilang ng mga qubit | Initialization        | Error     | Kabuuang oras (min) | Paggamit ng runtime (min) |
| ---------------------------- | ------------------- | --------------------- | --------- | ------------------- | ------------------------- |
| Inviscid Burgers' equation   | 50                  | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                  | 25                        |
| Hypoelastic 1D tensile test  | 18                  | `RANDOM`              | $10^{-2}$ | 123                 | 100                       |
## Magsimula
Punan ang [form para humiling ng access sa QUICK-PDE function](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Pagkatapos, kung nai-save mo na ang iyong account [sa iyong lokal na environment](/guides/functions#install-qiskit-functions-catalog-client), piliin ang function tulad ng sumusunod:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Mga Halimbawa

Para makapagsimula, subukan ang isa sa mga sumusunod na halimbawa:

### Computational Fluid Dynamics (CFD)

Kapag ang mga initial condition ay nakatakda sa $u(0,x) = x$, ang mga resulta ay tulad ng sumusunod:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Suriin ang [status](/guides/functions#check-job-status) ng iyong Qiskit Function workload o kunin ang mga [resulta](/guides/functions#retrieve-results) tulad ng sumusunod:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Material Deformation
Ang use case ng material deformation ay nangangailangan ng mga pisikal na parameter ng iyong materyal at ng inilapat na puwersa, tulad ng sumusunod:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

![Output ng nakaraang code cell](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Ang sumusunod ay isang halimbawa kung paano makuha ang halaga ng function para sa isang partikular na set ng mga coordinate:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



## Kunin ang mga mensahe ng error
Kung ang status ng iyong workload ay `ERROR`, gamitin ang `job.error_message()` para makuha ang mensahe ng error na makakatulong sa pag-debug, tulad ng sumusunod: